# Book files

The same portfolios as TOML. `[book]` names the book and can set a default volume. Each
`[[trades]]` entry has a `kind`. A `schedule` takes dates from the market instead of writing
them out, and a moneyness is `forward * moneyness`, rounded to two decimals, using the
forward on the settlement date. An Asian moneyness uses the average of the forwards on the
averaging dates.

`python/sjd/books/sample.toml` is the sample book: a 97/103 collar facing a producer, a
forward struck at 98% of the curve, and an Asian struck at 99%. A collar, seagull, or
participating forward takes `client = "producer"` or `"consumer"`. Omitted, it is a producer,
so the bank sells the put and buys the call.


In [ ]:
import logging
import math
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

import sjd
from sjd import book, examples

sjd.set_log_level("INFO")

market = examples.sample_market()
environment = market["environment"]
sjd.require_valid(environment)
sample_path = Path(examples.__file__).resolve().parent / "books" / "sample.toml"
sample_text = sample_path.read_text(encoding="utf-8")
print(sample_text)
assert "[book]" in sample_text
assert "[[trades]]" in sample_text
assert 'kind = "collar"' in sample_text
assert 'client = "producer"' in sample_text
assert 'schedule = "collar"' in sample_text
assert "put_moneyness" in sample_text


## Loading

`load_book` reads a path and `parse_book` reads text. Both return a `Book`. The sample file
and `examples.sample_contracts` are the same trades, so each instrument prices to the same
value.


In [ ]:
loaded = book.load_book(sample_path, market)
parsed = book.parse_book(sample_text, market)
reference = examples.sample_contracts(market["forward"], market)

assert loaded.name == "sample"
assert [trade.kind for trade in loaded.trades] == [
    "collar",
    "european_forward",
    "asian_forward",
]
assert len(loaded.contracts()) == 4
for contract in loaded.contracts() + reference["contracts"]:
    sjd.require_valid(contract, environment)

pairs = [
    (loaded.trades[0].contracts[0], reference["put"]),
    (loaded.trades[0].contracts[1], reference["call"]),
    (loaded.trades[1].contracts[0], reference["forward"]),
    (loaded.trades[2].contracts[0], reference["asian"]),
]
for left, right in pairs:
    assert type(left) is type(right)
    assert left.buy_sell == right.buy_sell
    assert math.isclose(left.get_value(environment), right.get_value(environment))
assert loaded.trades[0].contracts[0].buy_sell == sjd.BuySell.SELL
assert loaded.trades[0].contracts[1].buy_sell == sjd.BuySell.BUY
assert math.isclose(
    loaded.portfolio().get_value(environment),
    parsed.portfolio().get_value(environment),
)
print(f"sample book {loaded.portfolio().get_value(environment):.2f}")


## Every kind

One file uses each kind once. The `[book]` volume applies where a trade omits its own.
Strikes are absolute or a moneyness, and dates are a `schedule` or written out. The option
omits its premium date and amount, so the premium date is the settlement and the amount is
zero. `[margin]` becomes a `SimpleMargin` on the market anchor, and `portfolio()` is margined.


In [ ]:
settlement = market["collar_settlement"]
forward_settlement = market["forward_settlement"]
asian_start = market["averaging_dates"][0]
asian_end = market["averaging_dates"][-1]
asian_settlement = market["asian_settlement"]
absolute = round(market["forward"].get_forward(settlement), 2)
every_kind = f'''
[book]
name = "every kind"
volume = 100000

[[trades]]
kind = "cashflow"
settlement = {settlement.isoformat()}
buy_sell = "BUY"

[[trades]]
kind = "european_forward"
schedule = "forward"
strike_moneyness = 0.98
buy_sell = "BUY"

[[trades]]
kind = "european_option"
schedule = "collar"
strike = {absolute}
type = "CALL"
buy_sell = "BUY"

[[trades]]
kind = "asian_forward"
schedule = "asian"
strike_moneyness = 0.99
volume = 5
buy_sell = "BUY"

[[trades]]
kind = "asian_option"
averaging_dates = [{asian_start.isoformat()}, {asian_end.isoformat()}]
settlement = {asian_settlement.isoformat()}
strike_moneyness = 1.0
type = "PUT"
volume = 8
buy_sell = "SELL"

[[trades]]
kind = "collar"
client = "producer"
schedule = "collar"
put_moneyness = 0.97
call_moneyness = 1.03

[[trades]]
kind = "strip"
settlements = [{settlement.isoformat()}, {forward_settlement.isoformat()}]
strikes = [17.2, 17.6]
buy_sell = "BUY"

[[trades]]
kind = "seagull"
client = "producer"
schedule = "collar"
floor_strike = 16
put_strike = 17
call_strike = 18.5
volume = 1000

[[trades]]
kind = "participating_forward"
client = "producer"
schedule = "collar"
put_moneyness = 0.97
call_moneyness = 1.03
participation = 0.4

[margin]
use_my_threshold = true
my_threshold = 1000
use_cpty_threshold = true
cpty_threshold = 1000
minimum_transfer_amount = 100
rounding = 10
'''
full = book.parse_book(every_kind, market)
assert [trade.kind for trade in full.trades] == [
    "cashflow",
    "european_forward",
    "european_option",
    "asian_forward",
    "asian_option",
    "collar",
    "strip",
    "seagull",
    "participating_forward",
]
assert full.contracts()[0].volume == 100_000.0
assert full.trades[5].contracts[0].buy_sell == sjd.BuySell.SELL
assert full.trades[5].contracts[1].buy_sell == sjd.BuySell.BUY
assert full.trades[8].contracts[0].volume == 100_000.0
assert full.trades[8].contracts[0].buy_sell == sjd.BuySell.SELL
assert math.isclose(full.trades[8].contracts[1].volume, 60_000.0)
assert full.trades[8].contracts[1].buy_sell == sjd.BuySell.BUY
option = full.trades[2].contracts[0]
sjd.require_valid(option, environment)
assert math.isclose(option.get_value(environment), option.get_option_value(environment))
assert full.portfolio().is_margined()
assert full.margin.get_margin_call(10_000.0) == 9_000.0
priced = full.portfolio()
full_report = sjd.require_valid(priced, environment)
assert full_report.ok
print(f"every kind {priced.get_value(environment):.2f}")
full_report


## What can go wrong

`BookError` names the trade and the field. Loading checks the file. A book that loads is
still validated against the market with `require_valid`: that checks the trades.


In [ ]:
def refused(text, fragment, market=market):
    try:
        book.parse_book(text, market)
    except book.BookError as error:
        assert fragment in str(error)
        print(error)
        return
    raise AssertionError(fragment)

refused("this is not toml [[", "not valid TOML")
refused('[[trades]]\nkind = "swap"\n', "kind is not a known trade")
refused(
    f'''
[[trades]]
kind = "cashflow"
settlement = {settlement.isoformat()}
buy_sell = "BUY"
colour = "blue"
''',
    "unknown fields",
)
refused(
    f'''
[[trades]]
kind = "cashflow"
settlement = {settlement.isoformat()}
buy_sell = "BUY"
''',
    "volume is required",
)
refused(
    f'''
[[trades]]
kind = "european_forward"
settlement = {settlement.isoformat()}
strike = 17
strike_moneyness = 0.98
volume = 10
buy_sell = "BUY"
''',
    "exactly one of strike or strike_moneyness",
)
refused(
    f'''
[[trades]]
kind = "european_forward"
settlement = {settlement.isoformat()}
strike_moneyness = 0.98
volume = 10
buy_sell = "BUY"
''',
    "needs a market",
    market=None,
)
refused(
    f'''
[[trades]]
kind = "collar"
maturity = {market["collar_maturity"].isoformat()}
settlement = {settlement.isoformat()}
put_strike = 20
call_strike = 18
volume = 10
''',
    "put_strike must be below call_strike",
)
refused(
    f'''
[[trades]]
kind = "collar"
maturity = {market["collar_maturity"].isoformat()}
settlement = {settlement.isoformat()}
put_strike = 17
call_strike = 18
volume = 10
client = "dealer"
''',
    "must be producer or consumer",
)

checked = sjd.require_valid(loaded.portfolio(), environment)
assert checked.ok
checked


## Save and reload

The book text is written to a temporary file and loaded. The `Book` pickles, and loading
the file again prices to the same value. Nothing in the library writes a book back to TOML:
the text written here is the text the notebook composed.


In [ ]:
import pickle
import tempfile

scratch = tempfile.TemporaryDirectory()
path = Path(scratch.name) / "book.toml"
path.write_text(every_kind, encoding="utf-8")
from_file = book.load_book(path, market)
pickled = pickle.loads(pickle.dumps(from_file))
again = book.load_book(path, market)

assert math.isclose(
    from_file.portfolio().get_value(environment), priced.get_value(environment)
)
assert math.isclose(
    pickled.portfolio().get_value(environment), from_file.portfolio().get_value(environment)
)
assert math.isclose(
    again.portfolio().get_value(environment), from_file.portfolio().get_value(environment)
)
assert pickled.name == "every kind"
print("reloaded", path.name)
